In [1]:
import time
import requests
import numpy as np
import pandas as pd
from sgp4.api import jday
from sgp4.api import Satrec
from datetime import datetime, timezone
import matplotlib.pyplot as plt
from skyfield.api import EarthSatellite, load, wgs84
import geocoder
from pathlib import Path
from sklearn.metrics import confusion_matrix

In [2]:
def makeJulianDateNow():
    t = datetime.now()
    jd, fr = jday(t.year, t.month, t.day, t.hour, t.minute, t.second)

    return jd, fr


def getMyLocation():
    g = geocoder.ip('me')
    g = g.latlng

    lat = g[0]
    lon = g[1]

    return lat, lon


def satelliteTracker(dataFrame, sat_name, checkData=True, timer=10):
    run = True
    count = 0
    positions = []

    while run:
        # Get new data if needed
        if checkData:
            api_url = "https://api.spacexdata.com/v4/starlink"
            data = getData(api_url)

            # Make DataFrame
            satarr = []
            for d in data:
                satarr.append(d['spaceTrack'])

            sat_df = pd.DataFrame(satarr)

            checkData = False
        else:
            sat_df = dataFrame

        # Track a satellite
        positions.append(trackOverhead(sat_name, sat_df))

        if count <= timer:
            count += 1
        else:
            run = False
            return positions
        
        
def skyfieldTracker(dataFrame, sat_name, myCoordinates, timer=1, printData=False):
    run = True
    count = 0
    positions = []

    # Get data
    new_sat = dataFrame.loc[dataFrame['OBJECT_NAME'] == f'{sat_name}']
    line1, line2 = getTLE(new_sat)
    
    # Coordinates
    lat = myCoordinates[0]
    lon = myCoordinates[1]
    location = wgs84.latlon(lat, lon)

    # Timestamp
    ts = load.timescale()
    t = ts.now()
    timestamp = t.astimezone(timezone.utc)
    date = timestamp.strftime("%Y-%m-%d")
    utc_time = timestamp.strftime("%H:%M:%S")

    timestamp = f'{date} {utc_time}'
    
    satellite = EarthSatellite(line1, line2, sat_name, ts)

    difference = satellite - location

    # position of the satellite relative to observer in x, y, z values
    topocentric = difference.at(t)

    # altitude, azimuth, distance
    alt, az, distance = topocentric.altaz()

    if printData:
        print(f"Tracking {sat_name}")
        print("----------------------")
        print(" ")
        print(f'Time:  {timestamp}')
        print('Altitude: ', alt)
        print('Azimuth: ', az)
        print('Distance: {:.1f} km'.format(distance.km))
        print(" ")
        
        if alt.degrees > 0:
            print(f'{sat_name} is above the horizon')
        else: 
            print(f'{sat_name} is below the horizon')

    # Track a satellite
    positions.append([alt, az, distance.km, timestamp])

    if count <= timer:
        count += 1
    else:
        run = False

    return positions

In [3]:
jd, fr = makeJulianDateNow()
jd, fr

(2460664.5, 0.09921296296296296)

In [4]:
my_lat, my_lon = getMyLocation()
ground_station = (my_lat, my_lon)
ground_station

(33.2285, -97.1813)

In [5]:
weather_sats = pd.read_csv('weather_satellites.csv')
weather_sats

,sat_name,tle_line1,tle_line2
0,GOES-16,1 41866U 16071A 24354.79751793 -.00000241 0...,2 41866 0.0439 224.8146 0001664 88.7757 347...
1,GOES-17,1 43226U 18022A 24354.61373791 -.00000084 0...,2 43226 0.0400 172.9510 0003624 90.0312 301...
2,GOES-19,1 60133U 24119A 24354.79456399 -.00000176 0...,2 60133 0.0428 188.6740 0001319 123.2805 333...


In [6]:
def skyfieldTracker(line1, line2, sat_name, ground_station, printData=False):
    positions = []
    
    # Coordinates
    lat = ground_station[0]
    lon = ground_station[1]
    location = wgs84.latlon(lat, lon)

    # Timestamp
    ts = load.timescale()
    t = ts.now()
    timestamp = t.astimezone(timezone.utc)
    date = timestamp.strftime("%Y-%m-%d")
    utc_time = timestamp.strftime("%H:%M:%S")

    timestamp = f'{date} {utc_time}'
    
    satellite = EarthSatellite(line1, line2, sat_name, ts)

    difference = satellite - location

    # position of the satellite relative to observer in x, y, z values
    topocentric = difference.at(t)

    # altitude, azimuth, distance
    alt, az, distance = topocentric.altaz()

    if printData:
        print(f"Tracking {sat_name}")
        print("----------------------")
        print(" ")
        print(f'Time:  {timestamp}')
        print('Altitude: ', alt)
        print('Azimuth: ', az)
        print('Distance: {:.1f} km'.format(distance.km))
        print(" ")
        
        if alt.degrees > 0:
            print(f'{sat_name} is above the horizon')
        else: 
            print(f'{sat_name} is below the horizon')

    # Track a satellite
    positions.append([alt, az, distance.km, timestamp])

    return positions


In [7]:
sat = weather_sats.iloc[0]
sat_name = sat['sat_name']
line1 = sat['tle_line1']
line2 = sat['tle_line2']
ground_station


(33.2285, -97.1813)

In [ ]:
seconds_elapsed = 0
timer = 1
run_time = 10

while seconds_elapsed < run_time:
    positions = skyfieldTracker(line1, line2, sat_name, ground_station)
    print(positions)
    seconds_elapsed += 1
    time.sleep(timer)

In [9]:
positions[0][0].degrees

44.69142258683717

## Track overhead

`e` will be a non-zero error code if the satellite position could not be computed for the given date. You can from sgp4.api import SGP4_ERRORS to access a dictionary mapping error codes to error messages explaining what each code means.

`r` measures the satellite position in kilometers from the center of the earth in the idiosyncratic True Equator Mean Equinox coordinate frame used by SGP4.

` v` velocity is the rate at which the position is changing, expressed in kilometers per second.

- https://pypi.org/project/sgp4/

In [10]:
def trackOverhead(line1, line2, print_data=False):
    # Get sat position and velocity
    date = makeJulianDateNow()

    s = line1
    t = line2
    tracked_sat = Satrec.twoline2rv(s, t)

    jd, fr = date[0], date[1]
    
    e, r, v = tracked_sat.sgp4(jd, fr)
    
    if print_data:
        # True Equator Mean Equinox position (km)
        print("Position: ", r)
        # True Equator Mean Equinox velocity (km/s)
        print("Velocity: ", v)

    return [e, r, v]

In [11]:
sat_data = trackOverhead(line1, line2)

In [12]:
sat_data[1]

(27211.214184011198, 32209.6808884025, 14.571117106311416)

In [42]:
from pynput import keyboard

def on_press(key):
    try:
        k = key.char
        if k == 's':
            print("Save")
            
    except AttributeError:
        pass

def on_release(key):
    if key == keyboard.Key.esc:
        # Stop listener
        return False

# # Collect events until released
# with keyboard.Listener(
#         on_press=on_press,
#         on_release=on_release) as listener:
#     listener.join()

# ...or, in a non-blocking fashion:
listener = keyboard.Listener(
    on_press = on_press,
    on_release = on_release)

listener.start()

This process is not trusted! Input event monitoring will not be possible until it is added to accessibility clients.


alphanumeric key f pressed
Save
Save
Save
Save
Save
Save
Save
